# Boltz structure prediction

Select a GPU under **Runtime → Change runtime type**, then run the cells in order. This notebook installs Boltz 2.2.1, accepts one standard Boltz YAML file, and exposes all `boltz predict` options. See the [upstream input and prediction guide](https://github.com/jwohlwend/boltz/blob/v2.2.1/docs/prediction.md).

**Privacy:** the default MSA setting sends protein sequences to the public MMseqs2 service. Disable it if you cannot share sequences and provide MSAs in the YAML instead. Colab files are temporary unless saved elsewhere. First runs download model weights and may take substantial time and GPU memory.

In [ ]:
# Install the pinned CLI into a Python 3.12 virtual environment (Boltz 2.2.1 requires Python 3.10-3.12;
# Colab's default Python may be newer, so uv provisions a compatible interpreter).
import os
import subprocess
import sys

venv = '/content/boltz-venv'
venv_python = f'{venv}/bin/python'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', '--clear', venv], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', venv_python, 'boltz[cuda]==2.2.1'], check=True)
os.environ['PATH'] = f'{venv}/bin:' + os.environ['PATH']


def cuda_available():
    result = subprocess.run([venv_python, '-c', 'import torch; print(torch.cuda.is_available())'],
                            capture_output=True, text=True)
    return result.stdout.strip() == 'True'


print('CUDA available:', cuda_available())
if not cuda_available():
    print('No CUDA GPU detected. Select a GPU runtime or choose accelerator=cpu (much slower).')

## Upload a YAML file
Upload exactly one `.yaml` or `.yml` file. Boltz reads referenced MSA and template paths relative to `/content`, not the uploaded YAML's folder. Upload supporting files using Colab's Files pane or mount Drive, and use accessible paths in the YAML. Use the upstream schema without converting it.

In [ ]:
from pathlib import Path
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one Boltz YAML file.')
filename, contents = next(iter(uploaded.items()))
if Path(filename).name != filename or Path(filename).suffix.lower() not in ('.yaml', '.yml'):
    raise ValueError('The uploaded file must have a .yaml or .yml extension and a plain filename.')
input_dir = Path('/content/boltz_inputs')
input_dir.mkdir(parents=True, exist_ok=True)
input_path = input_dir / filename
input_path.write_bytes(contents)
print('Input:', input_path)

## Optional: mount Google Drive
Run this cell only if you need Drive for input files, checkpoints, the cache, or results. Set the relevant paths in the YAML or form below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Prediction options
These fields cover the complete Boltz 2.2.1 `predict` CLI. Optional text fields left blank are omitted, so Boltz supplies its own defaults. The default output directory and enabled MSA server are notebook conveniences; the remaining settings follow the CLI defaults. For authentication, select a method; passwords and API keys are prompted for at run time, not stored in the form. `override` recomputes predictions, but Boltz can still reuse preprocessed inputs: when changing the YAML, MSA, templates or preprocessing settings, use a new output directory or a new YAML filename.

In [ ]:
#@title Configure boltz predict
out_dir = '/content/boltz_outputs' #@param {type:"string"}
cache = '' #@param {type:"string"}
checkpoint = '' #@param {type:"string"}
affinity_checkpoint = '' #@param {type:"string"}
model = 'boltz2' #@param ['boltz2', 'boltz1']
devices = 1 #@param {type:"integer"}
accelerator = 'gpu' #@param ['gpu', 'cpu', 'tpu']
recycling_steps = 3 #@param {type:"integer"}
sampling_steps = 200 #@param {type:"integer"}
diffusion_samples = 1 #@param {type:"integer"}
max_parallel_samples = 5 #@param {type:"integer"}
step_scale = '' #@param {type:"string"}
write_full_pae = False #@param {type:"boolean"}
write_full_pde = False #@param {type:"boolean"}
output_format = 'mmcif' #@param ['mmcif', 'pdb']
num_workers = 2 #@param {type:"integer"}
override = False #@param {type:"boolean"}
seed = '' #@param {type:"string"}
use_msa_server = True #@param {type:"boolean"}
msa_server_url = 'https://api.colabfold.com' #@param {type:"string"}
msa_pairing_strategy = 'greedy' #@param {type:"string"}
msa_auth = 'none' #@param ['none', 'basic', 'api_key']
msa_server_username = '' #@param {type:"string"}
api_key_header = '' #@param {type:"string"}
use_potentials = False #@param {type:"boolean"}
method = '' #@param {type:"string"}
preprocessing_threads = '' #@param {type:"string"}
affinity_mw_correction = False #@param {type:"boolean"}
sampling_steps_affinity = 200 #@param {type:"integer"}
diffusion_samples_affinity = 5 #@param {type:"integer"}
max_msa_seqs = 8192 #@param {type:"integer"}
subsample_msa = False #@param {type:"boolean"}
num_subsampled_msa = 1024 #@param {type:"integer"}
no_kernels = False #@param {type:"boolean"}
write_embeddings = False #@param {type:"boolean"}

For the exact meanings, choices and defaults, run the help cell. In particular, set `no_kernels` if the GPU does not support Boltz's optional CUDA kernels.

In [ ]:
subprocess.run(['boltz', 'predict', '--help'], check=True)

In [ ]:
def build_command(input_file, options):
    """Build an argv list without shell interpolation or secret values."""
    value_options = {
        'out_dir': '--out_dir', 'cache': '--cache', 'checkpoint': '--checkpoint',
        'affinity_checkpoint': '--affinity_checkpoint', 'model': '--model',
        'devices': '--devices', 'accelerator': '--accelerator',
        'recycling_steps': '--recycling_steps', 'sampling_steps': '--sampling_steps',
        'diffusion_samples': '--diffusion_samples', 'max_parallel_samples': '--max_parallel_samples',
        'step_scale': '--step_scale', 'output_format': '--output_format',
        'num_workers': '--num_workers', 'seed': '--seed',
        'msa_server_url': '--msa_server_url', 'msa_pairing_strategy': '--msa_pairing_strategy',
        'msa_server_username': '--msa_server_username', 'api_key_header': '--api_key_header',
        'method': '--method', 'preprocessing_threads': '--preprocessing-threads',
        'sampling_steps_affinity': '--sampling_steps_affinity',
        'diffusion_samples_affinity': '--diffusion_samples_affinity',
        'max_msa_seqs': '--max_msa_seqs', 'num_subsampled_msa': '--num_subsampled_msa',
    }
    flag_options = {
        'write_full_pae': '--write_full_pae', 'write_full_pde': '--write_full_pde',
        'override': '--override', 'use_msa_server': '--use_msa_server',
        'use_potentials': '--use_potentials', 'affinity_mw_correction': '--affinity_mw_correction',
        'subsample_msa': '--subsample_msa', 'no_kernels': '--no_kernels',
        'write_embeddings': '--write_embeddings',
    }
    command = ['boltz', 'predict', str(input_file)]
    for name, flag in value_options.items():
        value = options[name]
        if value is not None and str(value).strip() != '':
            command.extend([flag, str(value)])
    for name, flag in flag_options.items():
        if options[name]:
            command.append(flag)
    return command

## Run prediction
The command is passed directly to Boltz without a shell. Credentials are supplied through Boltz's environment variables and are not printed as command-line arguments. The full Boltz logs and result files may contain sensitive input information.

In [ ]:
import getpass
import os
import re
import shlex

structures = []
if not input_path.is_file():
    raise FileNotFoundError('Upload the YAML file before running the prediction.')
if not out_dir.strip():
    raise ValueError('Set an output directory.')
if accelerator == 'gpu' and not cuda_available():
    raise RuntimeError('No CUDA GPU detected. Select a GPU runtime or set accelerator to cpu.')
if msa_auth not in ('none', 'basic', 'api_key'):
    raise ValueError('Unknown MSA authentication method.')
if msa_auth != 'none' and not use_msa_server:
    raise ValueError('MSA authentication requires use_msa_server.')
if msa_auth == 'basic' and not msa_server_username.strip():
    raise ValueError('Basic authentication requires msa_server_username.')
if msa_auth != 'basic' and msa_server_username.strip():
    raise ValueError('Set msa_server_username only for basic authentication.')
if msa_auth != 'api_key' and api_key_header.strip():
    raise ValueError('Set api_key_header only for API key authentication.')

env = os.environ.copy()
for key in ('BOLTZ_MSA_USERNAME', 'BOLTZ_MSA_PASSWORD', 'MSA_API_KEY_VALUE'):
    env.pop(key, None)
if msa_auth == 'basic':
    env['BOLTZ_MSA_PASSWORD'] = getpass.getpass('MSA server password: ')
    if not env['BOLTZ_MSA_PASSWORD']:
        raise ValueError('MSA server password cannot be empty.')
elif msa_auth == 'api_key':
    env['MSA_API_KEY_VALUE'] = getpass.getpass('MSA server API key: ')
    if not env['MSA_API_KEY_VALUE']:
        raise ValueError('MSA server API key cannot be empty.')

options = {name: globals()[name] for name in (
    'out_dir', 'cache', 'checkpoint', 'affinity_checkpoint', 'model', 'devices', 'accelerator',
    'recycling_steps', 'sampling_steps', 'diffusion_samples', 'max_parallel_samples', 'step_scale',
    'write_full_pae', 'write_full_pde', 'output_format', 'num_workers', 'override', 'seed',
    'use_msa_server', 'msa_server_url', 'msa_pairing_strategy', 'msa_server_username', 'api_key_header',
    'use_potentials', 'method', 'preprocessing_threads', 'affinity_mw_correction',
    'sampling_steps_affinity', 'diffusion_samples_affinity', 'max_msa_seqs', 'subsample_msa',
    'num_subsampled_msa', 'no_kernels', 'write_embeddings',
)}
command = build_command(input_path, options)
results_dir = (Path('/content') / Path(out_dir).expanduser()).resolve() / f'boltz_results_{input_path.stem}'
if results_dir.exists() and not override:
    print('Existing results may be reused. Enable override to recompute them.')
print('Running:', shlex.join(command), flush=True)
process = subprocess.Popen(command, cwd='/content', env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
processing_failed = False
prediction_failed = False
try:
    for line in process.stdout:
        print(line, end='', flush=True)
        if 'Failed to process ' in line:
            processing_failed = True
        match = re.search(r'Number of failed examples:\s*(\d+)', line)
        if match and int(match.group(1)) > 0:
            prediction_failed = True
    returncode = process.wait()
finally:
    del env
if returncode:
    raise subprocess.CalledProcessError(returncode, command)
if processing_failed:
    raise RuntimeError('Boltz skipped the input during preprocessing; see the error above.')
if prediction_failed:
    raise RuntimeError('Boltz reported failed predictions; see the error above.')
prediction_dir = results_dir / 'predictions' / input_path.stem
structures = list(prediction_dir.glob('*_model_*.cif')) + list(prediction_dir.glob('*_model_*.pdb'))
if not structures:
    raise RuntimeError(f'Boltz produced no structure in {prediction_dir}; inspect the logs above.')
print('Results:', results_dir)
print('Structures:', *structures, sep='\n')

## Download results
The ZIP includes the whole Boltz results directory, including confidence and any requested affinity or intermediate files. Download it before the runtime expires. Large archives can take time to create and download; Drive output is an alternative.

In [ ]:
import shutil

if not structures:
    raise RuntimeError('Run a successful prediction first.')
archive = shutil.make_archive(str(Path('/content') / results_dir.name), 'zip', root_dir=results_dir.parent, base_dir=results_dir.name)
print('Archive:', archive)
files.download(archive)